In [1]:
import os
import sys
import tensorflow as tf
from pyspark.sql import SparkSession

os.environ["TF_CPP_MIN_LOG_LEVEL"] = "1"

print("Python:", sys.version)
print("TensorFlow:", tf.__version__)
print("CPU:", tf.config.list_physical_devices("CPU"))

I0000 00:00:1777953011.484657      46 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1777953014.622352      46 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.


Python: 3.11.15 (main, Mar  3 2026, 09:26:23) [GCC 11.4.0]
TensorFlow: 2.21.0
CPU: [PhysicalDevice(name='/physical_device:CPU:0', device_type='CPU')]


In [2]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("1_Check_Environment")
    .master("spark://spark-master:7077")
    .config("spark.hadoop.fs.defaultFS", "hdfs://namenode:8020")
    
    # Quan trọng trong Docker: worker cần gọi ngược về driver trong notebook container
    .config("spark.driver.host", "notebook")
    .config("spark.driver.bindAddress", "0.0.0.0")
    
    # Giảm tài nguyên cho bước kiểm tra môi trường
    .config("spark.driver.memory", "1g")
    .config("spark.executor.memory", "2g")
    .config("spark.executor.cores", "1")
    .config("spark.cores.max", "2")
    .config("spark.executor.instances", "1")
    
    .config("spark.sql.shuffle.partitions", "16")
    .config("spark.ui.port", "4040")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("WARN")

print("Spark version:", spark.version)
print("Spark master:", spark.sparkContext.master)

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/05/05 03:50:20 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Spark version: 3.5.8
Spark master: spark://spark-master:7077


In [3]:
spark.range(10).count()

10

In [4]:
spark.sparkContext._jvm.org.apache.hadoop.fs.FileSystem.get(
    spark.sparkContext._jsc.hadoopConfiguration()
).listStatus(
    spark.sparkContext._jvm.org.apache.hadoop.fs.Path("hdfs://namenode:8020/")
)

JavaObject id=o51